# Person Network Explorer

DS 6105 | Summer 2026 — **Know Their Names Capstone**

Search for a person by name or key fragment, then explore their social/family network out to *n* degrees of separation.

- **Node colors** reflect racial classification in the original records (gold = Black, blue = White, green = Mixed, gray = unknown).
- **Edge colors** reflect the relationship predicate (see legend cell below).
- Hover over any node or edge for details.

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import ipywidgets as widgets
from IPython.display import display, HTML

In [ ]:
PERSON = pd.read_parquet('PERSON.parquet')
RELATION = pd.read_parquet('RELATION.parquet').reset_index()

# One directed edge per pair — keep the highest-count assertion when predicates differ
RELATION_g = (
    RELATION
    .sort_values('n_assertions', ascending=False)
    .drop_duplicates(subset=['subject_key', 'object_key'])
)

G = nx.from_pandas_edgelist(
    RELATION_g,
    source='subject_key',
    target='object_key',
    edge_attr=['predicate', 'n_assertions'],
    create_using=nx.DiGraph(),
)
print(f'{len(PERSON):,} persons  |  {G.number_of_nodes():,} graph nodes  |  {G.number_of_edges():,} directed edges')

In [ ]:
PREDICATE_COLORS = {
    'wasEnslavedBy':    '#e74c3c',
    'isParentOf':       '#2980b9',
    'isChildOf':        '#5dade2',
    'isSiblingOf':      '#27ae60',
    'isSpouseOf':       '#e91e63',
    'IsFatherOf':       '#1a5276',
    'IsMotherOf':       '#76448a',
    'isGrandParentOf':  '#148f77',
    'isGrandChildOf':   '#1abc9c',
    'isGrandfatherOf':  '#148f77',
    'isGrandmotherOf':  '#76d7c4',
    'isNiblingOf':      '#e67e22',
    'isPiblingOf':      '#ca6f1e',
    'isCousinOf':       '#f39c12',
    'isChildInLawOf':   '#8e44ad',
    'isParentInLawOf':  '#9b59b6',
    'isSiblingInLawOf': '#6c3483',
}

RACE_COLORS = {
    'b': '#f1c40f',
    'w': '#85c1e9',
    'm': '#82e0aa',
    'x': '#d7dbdd',
}

LS_LABELS = {'e': 'enslaved', 'f': 'free', 'x': 'unknown'}

# CSS injected into the pyvis HTML so vis.js tooltip newlines render as line breaks.
# vis.js 9.x sets tooltip content via innerText (not innerHTML), so we use \n
# separators and white-space:pre-line to display them properly.
_TOOLTIP_CSS = """<style>
div.vis-tooltip {
    white-space: pre-line !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 12px;
    line-height: 1.6;
    padding: 8px 12px;
    max-width: 300px;
    background: rgba(28, 28, 46, 0.96) !important;
    color: #ecf0f1 !important;
    border: 1px solid #556 !important;
    border-radius: 5px !important;
    box-shadow: 0 3px 10px rgba(0,0,0,0.5);
}
</style>"""


def parse_key(key):
    parts = key.split('-', 4)
    if len(parts) < 5:
        return dict(birth_year='?', legal_status='x', norm_race='x', gender='x', name=key)
    raw = parts[4].replace('_', ' ').strip()
    name = ' '.join(w.capitalize() for w in raw.split()) if raw not in ('x', '') else '(unnamed)'
    birth = parts[0] if parts[0] != 'xxxx' else '?'
    return dict(birth_year=birth, legal_status=parts[1], norm_race=parts[2], gender=parts[3], name=name)


def build_ego_graph(center_key, degrees):
    """Build ego network and return an srcdoc iframe."""
    ego = nx.ego_graph(G, center_key, radius=degrees, undirected=True)

    net = Network(
        height='600px', width='100%',
        directed=True,
        bgcolor='#1c1c2e',
        font_color='#ecf0f1',
        cdn_resources='in_line',
        notebook=False,
    )
    net.set_options("""{
        "physics": {
            "barnesHut": {
                "gravitationalConstant": -6000,
                "centralGravity": 0.3,
                "springLength": 130
            },
            "stabilization": {"iterations": 200}
        },
        "interaction": {"hover": true, "tooltipDelay": 80},
        "edges": {"smooth": {"type": "curvedCW", "roundness": 0.1}}
    }""")

    dist_map = nx.single_source_shortest_path_length(ego.to_undirected(), center_key)

    for node in ego.nodes():
        info = parse_key(node)
        dist = dist_map.get(node, degrees)

        if node == center_key:
            color, size = '#f39c12', 30
        else:
            color = RACE_COLORS.get(info['norm_race'], '#d7dbdd')
            size = max(8, 22 - dist * 4)

        p = PERSON.loc[node] if node in PERSON.index else None
        n_rels = int(p.n_relations) if p is not None else 0
        # vis.js 9.x uses innerText for string titles, so use \n (not <br>).
        # The injected CSS sets white-space:pre-line so newlines render as line breaks.
        title = (
            f"{info['name']}\n"
            f"Key: {node}\n"
            f"Birth: {info['birth_year']}  |  Status: {LS_LABELS.get(info['legal_status'], '?')}\n"
            f"Race: {info['norm_race'].upper()}  |  Gender: {info['gender'].upper()}\n"
            f"Mentions: {int(p.n_mentions) if p is not None else '-'}  |  Relations: {n_rels}"
        )
        net.add_node(
            node, label=info['name'], title=title,
            color={'background': color, 'border': '#888',
                   'highlight': {'background': '#fff', 'border': '#f39c12'}},
            size=size, font={'size': 11, 'color': '#ecf0f1'},
        )

    show_labels = ego.number_of_edges() < 60
    for u, v, data in ego.edges(data=True):
        pred = data.get('predicate', '')
        ec = PREDICATE_COLORS.get(pred, '#7f8c8d')
        net.add_edge(
            u, v, title=pred, color=ec,
            label=pred if show_labels else '',
            arrows='to', width=1.5,
            font={'size': 9, 'color': '#ccc', 'strokeWidth': 0},
        )

    raw_html = net.generate_html()
    raw_html = raw_html.replace('</head>', _TOOLTIP_CSS + '\n</head>', 1)

    srcdoc = raw_html.replace('&', '&amp;').replace('"', '&quot;')
    return HTML(
        f'<iframe srcdoc="{srcdoc}" '
        f'width="100%" height="630px" style="border:none;display:block"></iframe>'
    )

In [ ]:
edge_legend = [
    ('wasEnslavedBy', '#e74c3c'),
    ('isParentOf / IsFatherOf / IsMotherOf', '#2980b9'),
    ('isChildOf', '#5dade2'),
    ('isSiblingOf', '#27ae60'),
    ('isSpouseOf', '#e91e63'),
    ('isGrandParentOf / isGrandChildOf', '#148f77'),
    ('isNiblingOf / isPiblingOf', '#e67e22'),
    ('isCousinOf', '#f39c12'),
    ('in-law relations', '#8e44ad'),
]
node_legend = [
    ('Center person', '#f39c12'),
    ('Black (B)', '#f1c40f'),
    ('White (W)', '#85c1e9'),
    ('Mixed (M)', '#82e0aa'),
    ('Unknown race', '#d7dbdd'),
]

def swatch(color, shape='2px'):
    border_radius = '50%' if shape == 'circle' else '2px'
    return f'<span style="display:inline-block;width:13px;height:13px;background:{color};border-radius:{border_radius};margin-right:4px;vertical-align:middle"></span>'

edge_html = ''.join(swatch(c) + f'<span style="margin-right:14px;font-size:12px">{l}</span>' for l, c in edge_legend)
node_html = ''.join(swatch(c, 'circle') + f'<span style="margin-right:14px;font-size:12px">{l}</span>' for l, c in node_legend)

display(HTML(
    '<div style="padding:10px 14px;background:#f8f9fa;border-radius:6px;border:1px solid #dee2e6;line-height:2">'
    f'<b>Edge colors</b><br>{edge_html}<br><b>Node colors</b><br>{node_html}'
    '</div>'
))

In [ ]:
all_keys = sorted(G.nodes())

search_w = widgets.Text(
    placeholder='Type name or key fragment (min 2 chars)…',
    description='Search:',
    layout=widgets.Layout(width='460px'),
    style={'description_width': '58px'},
)
person_w = widgets.Select(
    options=[],
    description='Person:',
    rows=7,
    layout=widgets.Layout(width='620px'),
    style={'description_width': '58px'},
)
degree_w = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description='Degrees:',
    continuous_update=False,
    layout=widgets.Layout(width='360px'),
    style={'description_width': '62px'},
)
info_w = widgets.HTML(
    '<div style="padding:8px 12px;color:#888;font-style:italic">'
    'Search for a person above to begin.</div>'
)
graph_w = widgets.Output(
    layout=widgets.Layout(border='1px solid #dee2e6', border_radius='4px', min_height='40px')
)


def update_list(change):
    q = change['new'].strip().lower()
    # Unobserve person_w while mutating options+value to avoid firing redraw
    # multiple times (once on implicit value reset, once on explicit value set).
    person_w.unobserve(redraw, names='value')
    if len(q) < 2:
        person_w.options = []
        person_w.observe(redraw, names='value')
        return
    matches = [k for k in all_keys if q in k][:80]
    person_w.options = matches
    if matches:
        person_w.value = matches[0]
    person_w.observe(redraw, names='value')
    redraw()  # call exactly once after the list is settled


def redraw(_change=None):
    key = person_w.value
    if not key:
        return
    deg = degree_w.value

    if key in PERSON.index:
        p = PERSON.loc[key]
        info = parse_key(key)
        info_w.value = (
            '<div style="padding:8px 14px;background:#f0f3f4;border-radius:6px;font-size:13px">'
            f'<b style="font-size:15px">{info["name"]}</b> &nbsp;&nbsp;'
            f'Birth: <b>{info["birth_year"]}</b> &nbsp;|&nbsp; '
            f'Status: <b>{LS_LABELS.get(info["legal_status"], info["legal_status"])}</b> &nbsp;|&nbsp; '
            f'Race: <b>{info["norm_race"].upper()}</b> &nbsp;|&nbsp; '
            f'Gender: <b>{info["gender"].upper()}</b> &nbsp;|&nbsp; '
            f'Mentions: <b>{int(p.n_mentions)}</b> &nbsp;|&nbsp; '
            f'Relations: <b>{int(p.n_relations)}</b>'
            '</div>'
        )

    with graph_w:
        graph_w.clear_output(wait=True)
        if key not in G:
            display(HTML(
                f'<p style="padding:12px;color:#e67e22">'
                f'&#9888; <b>{key}</b> has no recorded relations in the graph.</p>'
            ))
        else:
            display(build_ego_graph(key, deg))


search_w.observe(update_list, names='value')
person_w.observe(redraw, names='value')
degree_w.observe(redraw, names='value')

display(widgets.VBox([
    widgets.HBox([search_w, degree_w]),
    person_w,
    info_w,
    graph_w,
]))